In [ ]:
# ==============================================================================
# CELL 1: DATA ENGINEERING FINAL (FIXED UPLOAD FLOW)
# ==============================================================================

from google.colab import files
import pandas as pd
import numpy as np
import os
import io
import re
import html

# 1. BERSIHKAN LINGKUNGAN
print("🧹 Membersihkan file lama...")
!rm *.xlsx *.csv 2>/dev/null

# 2. UPLOAD FILE (LANGKAH PERTAMA & WAJIB)
print("="*60)
print("📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS")
print("   (Total 12 File: 6 File Nilai Matkul + 6 File Logs)")
print("="*60)

uploaded = files.upload()

# 3. PROSES FILE (SETELAH UPLOAD)
score_frames = []
log_frames = []

print("\n📦 [PROCESS 1] Mengkategorikan & Membaca File...")

for fn in uploaded.keys():
    try:
        # Deteksi Jenis File berdasarkan Nama
        if "logs_" in fn.lower():
            print(f"   📂 LOGS: {fn}")
            df = pd.read_excel(io.BytesIO(uploaded[fn]))
            log_frames.append(df)
        else:
            print(f"   📊 SCORE: {fn}")
            # Force String untuk amankan koma
            df = pd.read_excel(io.BytesIO(uploaded[fn]), dtype=str, engine='openpyxl')
            score_frames.append(df)
    except Exception as e:
        print(f"   ❌ Gagal baca {fn}: {e}")

# Validasi
if not score_frames:
    raise ValueError("❌ ERROR: Tidak ada file skor/nilai yang terdeteksi! Cek nama file Anda.")

# 4. PENGGABUNGAN (CONCATENATE)
print("\n🔄 [PROCESS 2] Menggabungkan Data...")
score_df = pd.concat(score_frames, ignore_index=True)
logs_df = pd.concat(log_frames, ignore_index=True) if log_frames else pd.DataFrame()

print(f"✅ Data Tergabung: {len(score_df)} Baris Nilai, {len(logs_df)} Baris Logs.")

# 5. DATA CLEANING (SAPU JAGAT)
print("\n🧹 [PROCESS 3] Membersihkan Data Angka...")

def clean_indo_number(val):
    val_str = str(val).strip().lower()
    if val_str in ['nan', 'none', '', 'null', 'nat', '-']: return np.nan
    val_str = val_str.replace(',', '.')
    try:
        f = float(val_str)
        if f < 0: return 0
        if f > 100: return 100
        return f
    except: return np.nan

# Strategi Gabungan Kolom
score_df['grade_quiz'] = score_df['final_quiz_grade'].apply(clean_indo_number) if 'final_quiz_grade' in score_df.columns else np.nan

if 'assignmentscore' in score_df.columns:
    score_df['grade_assign'] = score_df['assignmentscore'].apply(clean_indo_number)
else:
    score_df['grade_assign'] = np.nan

if 'raw_quiz_score' in score_df.columns:
    score_df['grade_raw'] = score_df['raw_quiz_score'].apply(clean_indo_number)
else:
    score_df['grade_raw'] = np.nan

# Coalesce (Prioritas)
score_df['final_score_fixed'] = score_df['grade_quiz'].fillna(score_df['grade_assign']).fillna(score_df['grade_raw'])
score_df['final_quiz_grade'] = score_df['final_score_fixed']

# Fix ID & Name
score_df['userid'] = pd.to_numeric(score_df['userid'], errors='coerce').fillna(0).astype(int)
score_df = score_df[score_df['userid'] > 0]

if 'assignmentname' not in score_df.columns: score_df['assignmentname'] = np.nan
if 'quizname' not in score_df.columns: score_df['quizname'] = np.nan
score_df['quizname'] = score_df['quizname'].fillna(score_df['assignmentname']).fillna("Unknown Activity")

# 6. VERIFIKASI DATA (User 66745)
print("\n🔍 [VERIFIKASI DATA KRUSIAL]")
u_check = 66745
cek_df = score_df[score_df['userid'] == u_check]
if not cek_df.empty:
    avg_val = cek_df['final_quiz_grade'].mean()
    print(f"   👤 User {u_check} (Matdis/Logmat):")
    print(f"      - Rata-rata Nilai: {avg_val:.2f}")
    if avg_val > 0: print("      ✅ STATUS: AMAN. Data nilai terbaca.")
    else: print("      ⚠️ STATUS: MASIH 0. Cek file raw.")
else:
    print(f"   ⚠️ User {u_check} tidak ditemukan.")

# 7. AGGREGASI & LOGS
print("\n⚙️ [PROCESS 4] Agregasi & Logs Processing...")

score_agg = score_df.groupby("userid").agg(
    num_attempts = ("quiz_state", lambda x: (x.astype(str) == "finished").sum()) if 'quiz_state' in score_df.columns else ("userid", "count"),
    mean_score_pct = ("final_quiz_grade", "mean"),
    num_quizzes_taken = ("userid", "count")
).reset_index()
score_agg['mean_score_pct'] = score_agg['mean_score_pct'].round(2)

# Logs
logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')
def extract_userid(txt):
    if pd.isna(txt): return np.nan
    t = html.unescape(str(txt))
    m = re.search(r"id\s+'?(\d+)'?", t)
    return int(m.group(1)) if m else np.nan

logs_df['actor_userid'] = logs_df['Description'].apply(extract_userid)
logs_clean = logs_df.dropna(subset=['actor_userid']).copy()
logs_clean['actor_userid'] = logs_clean['actor_userid'].astype(int)

logs_agg = logs_clean.groupby("actor_userid").agg(
    engagement_score = ("Time_parsed", "size"),
    last_activity = ("Time_parsed", "max"),
    first_activity = ("Time_parsed", "min")
).reset_index().rename(columns={"actor_userid": "userid"})

logs_agg['active_days'] = (logs_agg['last_activity'] - logs_agg['first_activity']).dt.days
logs_agg['active_days'] = logs_agg['active_days'].replace(0, 1)
logs_agg['consistency_score'] = (logs_agg['engagement_score'] / logs_agg['active_days']).round(2)

# 8. MERGE & SAVE
print("🛠️ Final Merge...")
final_df = pd.merge(score_agg, logs_agg, on="userid", how="outer").fillna(0)

def get_category(score):
    if score >= 80: return "High"
    elif score >= 60: return "Medium"
    else: return "Low"
final_df['performance_category'] = final_df['mean_score_pct'].apply(get_category)

# SIMPAN KE GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7"
if not os.path.exists(save_path): os.makedirs(save_path)

print("\n💾 Menyimpan 4 File Wajib...")
# 1. Score Cleaned (Untuk Grafik & Nilai)
score_df.to_csv(f"{save_path}/merged_score_data_cleaned.csv", index=False)
# 2. User Features (Untuk ML & Dashboard)
final_df.to_csv(f"{save_path}/user_level_features_final_for_ML.csv", index=False)
# 3. Logs Cleaned (Untuk Jadwal Belajar Optimal - WAJIB ADA)
logs_clean.to_csv(f"{save_path}/merged_logs_data_cleaned.csv", index=False)

print("\n" + "="*50)
print("✅ DATA ENGINEERING SELESAI & SUKSES!")
print(f"📂 Folder Output: {save_path}")
print("   (File ke-4 'recommendation_engine.pkl' akan dibuat di tahap ML)")
print("="*50)

🧹 Membersihkan file lama...
📥 SILAKAN UPLOAD SEMUA FILE RAW (LOGS & NILAI) SEKALIGUS
   (Total 12 File: 6 File Nilai Matkul + 6 File Logs)


Saving LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-01PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx to LOGIKA MATEMATIKA IF-48-02PJJ [IZA].xlsx
Saving LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx to LOGIKA MATEMATIKA IF-48-03PJJ [LZD].xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1315-1419.xlsx
Saving logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx to logs_CAK1DAB3-IF-48-01PJJ_20251113-1354-1942.xlsx
Saving logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx to logs_CAK1DAB3-IF-48-03PJJ_20251113-1123-2136.xlsx
Saving logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx to logs_CAK1EAB3-IF-48-01PJJ_20251113-1049-1948.xlsx
Saving logs_CAK1EAB3-IF-48-02PJJ_20251113-1008-1572.xlsx to logs_CAK1EAB3-IF-48-02PJJ_20251113-1008-1572.xlsx
Saving logs_CAK1EAB3-IF-48-03PJJ_20251113-0921-1421.xlsx to logs_CAK1EAB3-IF-48-03PJJ_20251113-0921-1421.xlsx
Saving MATEMATIKA DISKRIT IF-48-01PJJ [DTO].xlsx to MATEMATIKA D

/tmp/ipython-input-1452670974.py:120: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  logs_df["Time_parsed"] = pd.to_datetime(logs_df["Time"], dayfirst=True, errors='coerce')


🛠️ Final Merge...
Mounted at /content/drive

💾 Menyimpan 4 File Wajib...

✅ DATA ENGINEERING SELESAI & SUKSES!
📂 Folder Output: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7
   (File ke-4 'recommendation_engine.pkl' akan dibuat di tahap ML)


In [ ]:
# ==============================================================================
# CELL 2: MACHINE LEARNING (RETRAIN)
# ==============================================================================
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

# 1. LOAD DATA BARU
save_path = "/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7"
input_file = os.path.join(save_path, "user_level_features_final_for_ML.csv")

print(f"🔄 Memuat data ML dari: {input_file}")
if not os.path.exists(input_file):
    print("❌ File tidak ditemukan! Jalankan DE dulu.")
else:
    df = pd.read_csv(input_file)
    print(f"✅ Data Loaded: {len(df)} Students")

    # 2. TRAINING
    features = ['mean_score_pct', 'engagement_score', 'consistency_score', 'num_attempts']
    X = df[features].copy()
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df['ml_cluster'] = kmeans.fit_predict(X_scaled)

    print(f"📊 Silhouette Score: {silhouette_score(X_scaled, df['ml_cluster']):.4f}")

    # 3. LABELING
    cluster_summary = df.groupby('ml_cluster')[features].mean()
    def get_cluster_label(row):
        score = row['mean_score_pct']
        engage = row['engagement_score']
        if score > 70: return "High Performer (Star)"
        elif engage > 5000:
            if score > 50: return "Active Learner"
            else: return "Hard Worker / Struggling"
        elif engage > 1000: return "Balanced Learner"
        else: return "At Risk / Passive"

    cluster_labels = {}
    print("\n=== CLUSTER INTERPRETATION ===")
    for cluster_id, row in cluster_summary.iterrows():
        label = get_cluster_label(row)
        cluster_labels[cluster_id] = label
        print(f"Cluster {cluster_id} -> {label} (Avg Score: {row['mean_score_pct']:.1f})")

    # 4. SAVE & SYNC
    df['cluster'] = df['ml_cluster']
    df.to_csv(input_file, index=False) # Update CSV User

    model_data = {
        "model": kmeans, "scaler": scaler,
        "cluster_labels": cluster_labels, "features": features
    }
    pkl_path = os.path.join(save_path, "recommendation_engine.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(model_data, f)

    print(f"\n✅ ML SELESAI. Model tersimpan di '{pkl_path}'")
    print("\n⬇️ SILAKAN DOWNLOAD 4 FILE INI DARI DRIVE DAN PASANG DI BACKEND:")
    print("   1. merged_score_data_cleaned.csv")
    print("   2. user_level_features_final_for_ML.csv")
    print("   3. merged_logs_data_cleaned.csv")
    print("   4. recommendation_engine.pkl")

🔄 Memuat data ML dari: /content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7/user_level_features_final_for_ML.csv
✅ Data Loaded: 444 Students
📊 Silhouette Score: 0.7847

=== CLUSTER INTERPRETATION ===
Cluster 0 -> At Risk / Passive (Avg Score: 0.5)
Cluster 1 -> Hard Worker / Struggling (Avg Score: 5.6)
Cluster 2 -> Active Learner (Avg Score: 68.6)
Cluster 3 -> Balanced Learner (Avg Score: 59.2)

✅ ML SELESAI. Model tersimpan di '/content/drive/MyDrive/Semester 7 /Computing project /FINAL_FIX_V7/recommendation_engine.pkl'

⬇️ SILAKAN DOWNLOAD 4 FILE INI DARI DRIVE DAN PASANG DI BACKEND:
   1. merged_score_data_cleaned.csv
   2. user_level_features_final_for_ML.csv
   3. merged_logs_data_cleaned.csv
   4. recommendation_engine.pkl
